# FashionCLIP Metric Learning (triplet pipeline)

Этот ноутбук делает **дообучение только image-энкодера FashionCLIP** в парадигме metric learning.


In [ ]:
import os, cv2, math, time, random, pickle
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import roc_auc_score, average_precision_score


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

@dataclass
class Config:
    triplets_train_path: str = "data/train_triplets.parquet"
    triplets_val_path: str = "data/val_triplets.parquet"
    val_pairs_path: Optional[str] = "data/val_pairs.parquet"
    hf_model_name: str = "patrickjohncyh/fashion-clip"
    output_dir: str = "artifacts/fashion_clip_metric"
    cache_path: str = "artifacts/fashion_clip_metric/image_cache.pkl"
    rebuild_cache: bool = False
    batch_size: int = 64
    num_workers: int = 8
    prefetch_factor: int = 4
    persistent_workers: bool = True
    lr: float = 2e-5
    weight_decay: float = 1e-4
    epochs: int = 6
    trainable_vision_layers: int = 4
    use_projection_head: bool = False
    projection_dim: int = 512
    loss_names: Tuple[str, ...] = ("triplet", "softplus_triplet", "circle")
    margin: float = 0.2
    gamma: float = 80.0
    circle_m: float = 0.25
    use_amp: bool = True
    max_grad_norm: float = 1.0

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(cfg)
print("device:", device)


In [ ]:
def load_table(path: str) -> pd.DataFrame:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(path)
    if p.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)

train_triplets = load_table(cfg.triplets_train_path)
val_triplets = load_table(cfg.triplets_val_path)


In [ ]:
def collect_unique_paths(*dfs: pd.DataFrame) -> List[str]:
    paths = set()
    for df in dfs:
        for c in ["anchor_path", "pos_path", "neg_path"]:
            if c in df.columns:
                paths.update(df[c].astype(str).tolist())
    return sorted(paths)

def build_image_cache(paths: List[str], verbose: bool = True) -> Dict[str, np.ndarray]:
    cache = {}
    it = tqdm(paths, desc="Caching images") if verbose else paths
    for p in it:
        img_bgr = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(p)
        cache[str(p)] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return cache

def load_or_build_cache(cache_path: str, all_paths: List[str], rebuild: bool = False):
    os.makedirs(Path(cache_path).parent, exist_ok=True)
    if (not rebuild) and Path(cache_path).exists():
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    cache = build_image_cache(all_paths, verbose=True)
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    return cache

all_paths = collect_unique_paths(train_triplets, val_triplets)
IMAGE_CACHE = load_or_build_cache(cfg.cache_path, all_paths, rebuild=cfg.rebuild_cache)


In [ ]:
class TripletImageDataset(Dataset):
    def __init__(self, triplet_df: pd.DataFrame, image_cache: Dict[str, np.ndarray]):
        self.df = triplet_df.reset_index(drop=True)
        self.image_cache = image_cache
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.image_cache[str(row.anchor_path)], self.image_cache[str(row.pos_path)], self.image_cache[str(row.neg_path)]

def make_collate(processor: CLIPProcessor):
    def collate_fn(batch):
        a_imgs, p_imgs, n_imgs = zip(*batch)
        a = processor(images=list(a_imgs), return_tensors="pt")["pixel_values"]
        p = processor(images=list(p_imgs), return_tensors="pt")["pixel_values"]
        n = processor(images=list(n_imgs), return_tensors="pt")["pixel_values"]
        return a, p, n
    return collate_fn

processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)
collate_fn = make_collate(processor)
train_loader = DataLoader(TripletImageDataset(train_triplets, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=True, drop_last=True, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)
val_loader = DataLoader(TripletImageDataset(val_triplets, IMAGE_CACHE), batch_size=cfg.batch_size, shuffle=False, drop_last=False, num_workers=cfg.num_workers, pin_memory=torch.cuda.is_available(), collate_fn=collate_fn)


In [ ]:
class FashionCLIPMetricModel(nn.Module):
    def __init__(self, model_name: str, trainable_vision_layers: int = 4, use_projection_head: bool = False, projection_dim: int = 512):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        for p in self.clip.parameters():
            p.requires_grad = False
        for p in self.clip.visual_projection.parameters():
            p.requires_grad = True
        layers = self.clip.vision_model.encoder.layers
        for i in range(max(0, len(layers)-trainable_vision_layers), len(layers)):
            for p in layers[i].parameters():
                p.requires_grad = True
        emb_dim = self.clip.config.projection_dim
        self.head = nn.Identity() if not use_projection_head else nn.Sequential(nn.Linear(emb_dim, emb_dim), nn.GELU(), nn.Dropout(0.1), nn.Linear(emb_dim, projection_dim))

    def encode(self, pixel_values: torch.Tensor) -> torch.Tensor:
        z = self.head(self.clip.get_image_features(pixel_values=pixel_values))
        return F.normalize(z, p=2, dim=-1)

    def forward(self, a_px, p_px, n_px):
        return self.encode(a_px), self.encode(p_px), self.encode(n_px)


In [ ]:
def pairwise_cos(a, b):
    return (F.normalize(a, dim=-1) * F.normalize(b, dim=-1)).sum(dim=-1)

def triplet_loss(za, zp, zn, margin=0.2):
    return F.triplet_margin_loss(za, zp, zn, margin=margin, p=2)

def softplus_triplet_loss(za, zp, zn):
    d_ap = torch.norm(za - zp, p=2, dim=-1)
    d_an = torch.norm(za - zn, p=2, dim=-1)
    return F.softplus(d_ap - d_an).mean()

def circle_triplet_loss(za, zp, zn, m=0.25, gamma=80.0):
    s_p = pairwise_cos(za, zp)
    s_n = pairwise_cos(za, zn)
    alpha_p = torch.relu(1 + m - s_p)
    alpha_n = torch.relu(s_n + m)
    logits = gamma * (alpha_n * (s_n - m) - alpha_p * (s_p - (1 - m)))
    return F.softplus(logits).mean()

def compute_loss(loss_name, za, zp, zn, cfg):
    if loss_name == "triplet":
        return triplet_loss(za, zp, zn, margin=cfg.margin)
    if loss_name == "softplus_triplet":
        return softplus_triplet_loss(za, zp, zn)
    if loss_name == "circle":
        return circle_triplet_loss(za, zp, zn, m=cfg.circle_m, gamma=cfg.gamma)
    raise ValueError(loss_name)


In [ ]:
@torch.no_grad()
def triplet_batch_acc(za, zp, zn):
    return (pairwise_cos(za, zp) > pairwise_cos(za, zn)).float().mean().item()

def run_epoch(model, loader, optimizer, scaler, loss_name, train):
    model.train(train)
    losses, accs = [], []
    for a_px, p_px, n_px in tqdm(loader, leave=False):
        a_px, p_px, n_px = a_px.to(device), p_px.to(device), n_px.to(device)
        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and device.type == "cuda")):
                za, zp, zn = model(a_px, p_px, n_px)
                loss = compute_loss(loss_name, za, zp, zn, cfg)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
        losses.append(loss.item())
        accs.append(triplet_batch_acc(za.detach(), zp.detach(), zn.detach()))
    return {"loss": float(np.mean(losses)), "triplet_acc": float(np.mean(accs))}


In [ ]:
all_histories, best_ckpts = {}, {}
for loss_name in cfg.loss_names:
    model = FashionCLIPMetricModel(cfg.hf_model_name, cfg.trainable_vision_layers, cfg.use_projection_head, cfg.projection_dim).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and device.type == "cuda"))
    history, best_val = [], float("inf")
    best_path = os.path.join(cfg.output_dir, f"best_{loss_name}.pt")

    for epoch in range(1, cfg.epochs + 1):
        tr = run_epoch(model, train_loader, optimizer, scaler, loss_name=loss_name, train=True)
        va = run_epoch(model, val_loader, optimizer, scaler, loss_name=loss_name, train=False)
        history.append({"loss_name": loss_name, "epoch": epoch, "train_loss": tr["loss"], "val_loss": va["loss"], "train_triplet_acc": tr["triplet_acc"], "val_triplet_acc": va["triplet_acc"]})
        print(f"[{loss_name}] E{epoch:02d} tr_loss={tr['loss']:.4f} va_loss={va['loss']:.4f} tr_acc={tr['triplet_acc']:.4f} va_acc={va['triplet_acc']:.4f}")
        if va["loss"] < best_val:
            best_val = va["loss"]
            torch.save({"model_state": model.state_dict(), "config": cfg.__dict__, "loss_name": loss_name}, best_path)
    all_histories[loss_name] = pd.DataFrame(history)
    best_ckpts[loss_name] = best_path


In [ ]:
def pick_best_loss_by_val_triplet_acc(histories):
    best_name, best_metric = None, -1.0
    for name, h in histories.items():
        metric = float(h["val_triplet_acc"].max())
        if metric > best_metric:
            best_name, best_metric = name, metric
    return best_name

best_loss_name = pick_best_loss_by_val_triplet_acc(all_histories)
best_model = FashionCLIPMetricModel(cfg.hf_model_name, cfg.trainable_vision_layers, cfg.use_projection_head, cfg.projection_dim).to(device)
best_model.load_state_dict(torch.load(best_ckpts[best_loss_name], map_location=device)["model_state"])
best_model.eval()

@torch.no_grad()
def encode_image_paths(model, processor, image_paths, image_cache, batch_size=256):
    model.eval()
    vectors = []
    for i in tqdm(range(0, len(image_paths), batch_size), leave=False):
        chunk = image_paths[i:i+batch_size]
        imgs = [image_cache[str(p)] for p in chunk]
        px = processor(images=imgs, return_tensors="pt")["pixel_values"].to(device)
        vectors.append(model.encode(px).cpu())
    return torch.cat(vectors, dim=0)


## Инференс

1) Получаем эмбеддинги товаров через `encode_image_paths`.  
2) Считаем cosine: `(z_i * z_j).sum()`, так как векторы нормированы.  
3) Порог подбираем по валидации.

В этом ноутбуке реализованы 3 варианта лосса (включая **Circle Loss**) и переиспользован triplet-пайплайн с RAM-кешем изображений.


## Подробные пояснения по архитектуре и обучению

### Что именно дообучаем
- Основа — `patrickjohncyh/fashion-clip`.
- Обучаем только image-ветку: `visual_projection` и последние блоки `vision_model.encoder.layers`.
- Text-часть CLIP не используется, чтобы задача оставалась чисто image-only.

### Почему cosine на инференсе
- Модель возвращает L2-нормированные векторы, поэтому скалярное произведение эквивалентно cosine similarity.
- Это удобно для продакшна: эмбеддинги айтемов можно посчитать офлайн один раз, а затем быстро сравнивать любые пары.

### Лоссы
1. **Triplet margin loss** — базовый стабильный старт.
2. **Softplus-triplet** — более гладкий вариант без жёсткого margin-обрезания.
3. **Circle loss** — управляет вкладом сложных/лёгких примеров через `alpha_p/alpha_n`, часто даёт более чёткое разделение в embedding-space.

### Кэширование изображений
- Все уникальные пути из triplets поднимаются в RAM (`IMAGE_CACHE`), чтобы убрать bottleneck чтения диска.
- Кэш сохраняется в `pickle`, чтобы повторный запуск не тратил время на декодирование тех же картинок.

### Как использовать результат
1. Выбрать лучший чекпоинт по `val_triplet_acc` (или по pair-level AUC, если есть `val_pairs`).
2. Посчитать эмбеддинги каталога.
3. Для пары айтемов считать cosine.
4. Порог бинарного решения подобрать на валидации под бизнес-метрику (precision/recall/F1, либо cost-aware метрика).

### Идеи для следующего шага
- Hard negative mining (online/offline).
- Увеличить разнообразие позитивов по стилям/категориям.
- Добавить TTA на инференсе и/или EMA весов при обучении.
- Калибровать порог отдельно для разных категорий одежды.
